## SRP323114

**paper:** [PMID: 35099536](https://pmc.ncbi.nlm.nih.gov/articles/PMC8844503/) - Molecular Evolution across Mouse Spermatogenesis, 2022

**date, curator:** 2026-04-23, Sara Carsanaro

**notes**
* this is not single cell seq - this is facs sorting by cell type but then the cells are pooled
* not clear which Agilent SureSelect kit was used for library prep, but it is polyA and full_length because they all are
* for cell types - 
    * leptotene/zygotene = early meiotic spermatocytes
    * round spermatid = postmeiotic round spermatids

### annotation summary

In [22]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,LZ,CL:0000656,primary spermatocyte,perfect match
4,RS,CL:0000018,spermatid,perfect match


In [23]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,95 dpp,MmusDv:0000159,13-week-old stage,perfect match
7,96 dpp,MmusDv:0000159,13-week-old stage,perfect match
13,94 dpp,MmusDv:0000159,13-week-old stage,perfect match
16,101 dpp,MmusDv:0000160,14-week-old stage,perfect match
19,102 dpp,MmusDv:0000160,14-week-old stage,perfect match
25,80 dpp,MmusDv:0000157,11-week-old stage,perfect match
31,78 dpp,MmusDv:0000157,11-week-old stage,perfect match
35,77 dpp,MmusDv:0000157,11-week-old stage,perfect match
40,67 dpp,MmusDv:0000155,9-week-old stage,perfect match
42,68 dpp,MmusDv:0000155,9-week-old stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP323114"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████| 139/139 [02:10<00:00,  1.06it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX11086140,SRP323114,NextSeq 500,SRS9151654,,,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,,,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX11086139,SRP323114,NextSeq 500,SRS9151654,,,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,,,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX11086089,SRP323114,NextSeq 500,SRS9151654,,,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,,,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX11086035,SRP323114,NextSeq 500,SRS9151654,,,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,,,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX11086072,SRP323114,NextSeq 500,SRS9151635,,,MmusDv:0000159,13-week-old stage,,RS,95 dpp,,,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX11086019,SRP323114,NextSeq 500,SRS9151635,,,MmusDv:0000159,13-week-old stage,,RS,95 dpp,,,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX11086008,SRP323114,NextSeq 500,SRS9151635,,,MmusDv:0000159,13-week-old stage,,RS,95 dpp,,,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX11086051,SRP323114,NextSeq 500,SRS9151650,,,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,,,perfect match,M,BIK/g,,10092,,,,,3,BIK4665-2M-LZ,SAMN19597719,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX11086032,SRP323114,NextSeq 500,SRS9151650,,,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,,,perfect match,M,BIK/g,,10092,,,,,3,BIK4665-2M-LZ,SAMN19597719,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
9,SRX11086138,SRP323114,NextSeq 500,SRS9151656,,,MmusDv:0000159,13-week-old stage,,RS,96 dpp,,,perfect match,M,BIK/g,,10092,,,,,4,BIK4665-2M-RS,SAMN19597720,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-2M-RS,,,,,,TRANSCRIPTOMIC,cDNA


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['LZ' 'RS']


In [6]:

# conditional (based off infoOrgan)
library.loc[library["infoOrgan"] == "LZ", "anatId"] = "CL:0000656"
library.loc[library["infoOrgan"] == "LZ", "anatName"] = "primary spermatocyte"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "LZ", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "LZ", "anatBiologicalStatus"] = "not documented"

# conditional (based off infoOrgan)
library.loc[library["infoOrgan"] == "RS", "anatId"] = "CL:0000018"
library.loc[library["infoOrgan"] == "RS", "anatName"] = "spermatid"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "RS", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "RS", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX11086140,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX11086139,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX11086089,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX11086035,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX11086072,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX11086019,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX11086008,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX11086051,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,3,BIK4665-2M-LZ,SAMN19597719,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX11086032,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,3,BIK4665-2M-LZ,SAMN19597719,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
9,SRX11086138,SRP323114,NextSeq 500,SRS9151656,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,4,BIK4665-2M-RS,SAMN19597720,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells 

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['101 dpp' '102 dpp' '130 dpp' '148 dpp' '150 dpp' '62 dpp' '63 dpp'
 '65 dpp' '66 dpp' '67 dpp' '68 dpp' '69 dpp' '72 dpp' '73 dpp' '74 dpp'
 '75 dpp' '77 dpp' '78 dpp' '79 dpp' '80 dpp' '84 dpp' '87 dpp' '89 dpp'
 '93 dpp' '94 dpp' '95 dpp' '96 dpp']


In [8]:


# not a subspecies
library.loc[library["speciesId"] == "10096", "stageId"] = "UBERON:0018241"
library.loc[library["speciesId"] == "10096", "stageName"] = "prime adult stage"
# perfect match, missing child term, other
library.loc[library["speciesId"] == "10096", "stageAnnotationStatus"] = "other"

# not a subspecies
library.loc[library["speciesId"] == "10093", "stageId"] = "UBERON:0018241"
library.loc[library["speciesId"] == "10093", "stageName"] = "prime adult stage"
# perfect match, missing child term, other
library.loc[library["speciesId"] == "10093", "stageAnnotationStatus"] = "other"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX11086140,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX11086139,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX11086089,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX11086035,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX11086072,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX11086019,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX11086008,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX11086051,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,3,BIK4665-2M-LZ,SAMN19597719,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX11086032,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,3,BIK4665-2M-LZ,SAMN19597719,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
9,SRX11086138,SRP323114,NextSeq 500,SRS9151656,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,,,,,4,BIK4665-2M-RS,SAMN19597720,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells 

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'Agilent SureSelect'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX11086140,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX11086139,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX11086089,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX11086035,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX11086072,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX11086019,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX11086008,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX11086051,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,3,BIK4665-2M-LZ,SAMN19597719,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX11086032,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,3,BIK4665-2M-LZ,SAMN19597719,96,,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

      #libraryId       SRSId
57   SRX11086002  SRS9151631
56   SRX11086075  SRS9151631
55   SRX11086076  SRS9151631
54   SRX11086003  SRS9151632
53   SRX11086004  SRS9151632
52   SRX11086005  SRS9151632
48   SRX11086010  SRS9151634
49   SRX11086009  SRS9151634
50   SRX11086007  SRS9151634
4    SRX11086072  SRS9151635
5    SRX11086019  SRS9151635
6    SRX11086008  SRS9151635
39   SRX11086020  SRS9151644
38   SRX11086021  SRS9151644
37   SRX11086022  SRS9151644
35   SRX11086024  SRS9151645
36   SRX11086023  SRS9151645
95   SRX11086038  SRS9151647
97   SRX11086026  SRS9151647
96   SRX11086027  SRS9151647
94   SRX11086028  SRS9151648
93   SRX11086029  SRS9151648
92   SRX11086039  SRS9151648
91   SRX11086030  SRS9151649
90   SRX11086031  SRS9151649
89   SRX11086041  SRS9151649
7    SRX11086051  SRS9151650
8    SRX11086032  SRS9151650
33   SRX11086033  SRS9151651
32   SRX11086077  SRS9151651
31   SRX11086078  SRS9151651
30   SRX11086034  SRS9151652
28   SRX11086080  SRS9151652
29   SRX110860

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_1521/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [11]:
#library.loc[:,'sampleAge_value'] = ''
library.loc[:,'sampleAge_unit'] = 'day postpartum'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX11086140,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX11086139,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX11086089,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX11086035,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX11086072,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX11086019,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX11086008,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX11086051,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,3,BIK4665-2M-LZ,SAMN19597719,96,day postpartum,,,,,,,,,23/04/2026,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX11086032,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,3,BIK4665-2M-LZ,SAMN19597719,96,day p

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-23'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX11086140,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,,,,SAC,2026-04-23,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX11086139,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,,,,SAC,2026-04-23,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX11086089,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,,,,SAC,2026-04-23,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX11086035,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,,,,SAC,2026-04-23,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-1M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX11086072,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,,,,SAC,2026-04-23,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX11086019,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,,,,SAC,2026-04-23,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX11086008,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,,,,SAC,2026-04-23,cDNA sequencing from RNA extracted from FACS sorted cells (round spermatids),,BIK4665-1M-RS,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX11086051,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,3,BIK4665-2M-LZ,SAMN19597719,96,day postpartum,,,,,,,,SAC,2026-04-23,cDNA sequencing from RNA extracted from FACS sorted cells (leptotene/zygotene),,BIK4665-2M-LZ,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX11086032,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,3,BIK4665-2M-

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 35099536'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX11086140,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
1,SRX11086139,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
2,SRX11086089,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
3,SRX11086035,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
4,SRX11086072,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
5,SRX11086019,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
6,SRX11086008,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
7,SRX11086051,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,3,BIK4665-2M-LZ,SAMN19597719,96,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
8,SRX11086032,SRP323114,NextSeq 500,SRS9151650,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,3,BIK4665-2M-LZ,SAMN19597719,96,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
9,SRX11086138,SRP323114,NextSeq 500,SRS9151656,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,96 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,4,BIK4665-2M-RS,SAMN19597720,96,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP323114,RNAseq of FACS sorted testes from Mus wild-derived inbred strains,"We investigated the evolution of gene expression across spermatogenesis using transcriptomic data from cell populations sorted from mouse whole testes. RNAseq data are from four mouse species or subspecies: Mus musculus musculus, Mus musculus domesticus, Mus spretus, and Mus pahari.",SRA,,,,,,,PRJNA735780,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

139

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP323114,RNAseq of FACS sorted testes from Mus wild-derived inbred strains,"We investigated the evolution of gene expression across spermatogenesis using transcriptomic data from cell populations sorted from mouse whole testes. RNAseq data are from four mouse species or subspecies: Mus musculus musculus, Mus musculus domesticus, Mus spretus, and Mus pahari.",SRA,total,Bgee 1K,139,Agilent SureSelect,full_length,,PRJNA735780,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '35099536'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC8844503/'
experiment.loc[:,'DOI'] = '10.1093/molbev/msac023'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP323114,RNAseq of FACS sorted testes from Mus wild-derived inbred strains,"We investigated the evolution of gene expression across spermatogenesis using transcriptomic data from cell populations sorted from mouse whole testes. RNAseq data are from four mouse species or subspecies: Mus musculus musculus, Mus musculus domesticus, Mus spretus, and Mus pahari.",SRA,total,Bgee 1K,139,Agilent SureSelect,full_length,,PRJNA735780,35099536,https://pmc.ncbi.nlm.nih.gov/articles/PMC8844503/,10.1093/molbev/msac023,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 139
Errors: 139
Warnings: 0
Top codes:
  - BAD_VALUE: 139


#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
62308,ERX9575270,ERP139798,Illumina HiSeq 4000,ERS12565235,UBERON:0000082,adult mammalian kidney,MmusDv:0000001,life cycle,,whole kidney,NA,perfect match,full sampling,perfect match,NA,,,10090,TruSeq RNA Library Prep Kit,full_length,polyA,,,SHPC55,SAMEA110467207,,,,,,,PMID: 41565469,,,SAC,2026-04-23
62309,ERX9575271,ERP139798,Illumina HiSeq 4000,ERS12565236,UBERON:0000082,adult mammalian kidney,UBERON:0000104,life cycle,,whole kidney,NA,perfect match,full sampling,perfect match,NA,,,10129,TruSeq RNA Library Prep Kit,full_length,polyA,,,SHPC56,SAMEA110467208,,,,,,,PMID: 41565469,,,SAC,2026-04-23
62310,SRX11086140,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
62311,SRX11086139,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
62312,SRX11086089,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
62313,SRX11086035,SRP323114,NextSeq 500,SRS9151654,CL:0000656,primary spermatocyte,MmusDv:0000159,13-week-old stage,,LZ,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,1,BIK4665-1M-LZ,SAMN19597717,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23
62314,SRX11086072,SRP323114,NextSeq 500,SRS9151635,CL:0000018,spermatid,MmusDv:0000159,13-week-old stage,,RS,95 dpp,perfect match,not documented,perfect match,M,BIK/g,,10092,Agilent SureSelect,full_length,polyA,,2,BIK4665-1M-RS,SAMN19597718,95,day postpartum,,,,,PMID: 35099536,,,SAC,2026-04-23


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1203,ERP001989,Slc25a21_expression_profile,omozygosity for Slc25a21<tm1a(KOMP)Wtsi> resul...,SRA,partial,Bgee 1K,32,TruSeq RNA Sample Preparation Kit,full_length,,PRJEB672,24642684,https://pmc.ncbi.nlm.nih.gov/articles/PMC3958370/,10.1371/journal.pone.0091807,,"removed knockout samples, kept wild type only"
1204,ERP139798,Convergent adaptation to xeric environments in...,Our study aims at better understanding the mol...,SRA,total,Bgee 1K,57,TruSeq RNA Library Prep Kit,full_length,,PRJEB54931,41565469,https://pmc.ncbi.nlm.nih.gov/articles/PMC12951...,10.1101/gr.280089.124,,
1205,SRP323114,RNAseq of FACS sorted testes from Mus wild-der...,We investigated the evolution of gene expressi...,SRA,total,Bgee 1K,139,Agilent SureSelect,full_length,,PRJNA735780,35099536,https://pmc.ncbi.nlm.nih.gov/articles/PMC8844503/,10.1093/molbev/msac023,,


### add annotations to git

In [27]:
! git pull

Already up to date.


In [28]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [29]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../SRP000401/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [30]:
! git add $git_experiment_path $git_library_path

In [31]:
! git commit -m $commit_message_exp

[develop 710d1f2] adding annotated bulk experiment SRP323114
 2 files changed, 140 insertions(+)


In [32]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 4.36 KiB | 4.36 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   f036808..710d1f2  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push